In [1]:
from pathlib import Path

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

SAMOKAT_MAIN_PATH = DATA_DIR / "samokat_esci.csv"


In [2]:
import pandas as pd

df = pd.read_csv(SAMOKAT_MAIN_PATH, encoding='utf-8')

preprocessing = df.copy()
preprocessing['query'] = preprocessing['query'].str.lower().str.replace(
    r'[^\p{L}\s-]', '', regex=True
)


cобираю датасет: названия товаров + размеченные запросы

In [ ]:
def build_dataset(df, key_col, category_cols=('category1_name',), unique_category_only=False):
    category_cols = list(category_cols)
    d = df.dropna(subset=category_cols)
    if unique_category_only:
        nunique = d.groupby(key_col)[category_cols].nunique()
        clean_keys = nunique[(nunique == 1).all(axis=1)].index
        d = d[d[key_col].isin(clean_keys)]
    d = d.drop_duplicates(subset=[key_col])[[key_col] + category_cols].rename(columns={key_col: 'query'})
    return d

item_names = build_dataset(preprocessing, 'item_name', category_cols=('category1_name', 'category2_name'))

rows = preprocessing[preprocessing['final_answer'].isin(['e', 's'])]
category2_dataset = build_dataset(
    rows, 'query', category_cols=('category1_name', 'category2_name'), unique_category_only=True
)

category_dataset = pd.concat([category2_dataset, item_names], ignore_index=True)

print(f'всего строк: {len(category_dataset)}')
print(f'уникальных category1: {category_dataset["category1_name"].nunique()}')
print(f'уникальных category2: {category_dataset["category2_name"].nunique()}')


всего строк: 39209
уникальных category1: 59
уникальных category2: 288


In [ ]:
pair_counts = category_dataset.groupby(['category1_name', 'category2_name']).size()
valid_pairs = set(pair_counts[pair_counts >= 3].index)

category_dataset = category_dataset[
    category_dataset.apply(lambda r: (r['category1_name'], r['category2_name']) in valid_pairs, axis=1)
]

cat1_counts = category_dataset['category1_name'].value_counts()
valid_cat1 = cat1_counts[cat1_counts >= 2].index
category_dataset = category_dataset[category_dataset['category1_name'].isin(valid_cat1)]

print(f'после фильтрации: {len(category_dataset)} строк, {category_dataset["category1_name"].nunique()} category1')


после фильтрации: 39172 строк, 58 category1


In [11]:
from sklearn.model_selection import train_test_split

X = category_dataset['query']
y1 = category_dataset['category1_name']
y2 = category_dataset['category2_name']

X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X, y1, y2, test_size=0.15, random_state=42, stratify=y1,
)

print(f'train: {len(X_train)}, test: {len(X_test)}')


train: 33296, test: 5876


эмбеддинги

In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('d0rj/e5-small-en-ru')

X_train_prefixed = ["query: " + q for q in X_train]
X_test_prefixed = ["query: " + q for q in X_test]

X_train_emb = embedder.encode(X_train_prefixed, convert_to_numpy=True, show_progress_bar=True)
X_test_emb = embedder.encode(X_test_prefixed, convert_to_numpy=True, show_progress_bar=True)


/opt/anaconda3/envs/study/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Batches:   0%|          | 0/1041 [00:00<?, ?it/s]

Batches:   0%|          | 0/184 [00:00<?, ?it/s]

первая ступень: определение наиболее общей категории

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

kn_model = KNeighborsClassifier(weights='distance', n_neighbors=3, metric='euclidean')
kn_model.fit(X_train_emb, y1_train)

y1_pred = kn_model.predict(X_test_emb)

print(f'category1 accuracy: {accuracy_score(y1_test, y1_pred):.1%}')
print(f'category1 f1_macro: {f1_score(y1_test, y1_pred, average="macro"):.3f}')


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


category1 accuracy: 90.4%
category1 f1_macro: 0.846


ступень 2: определение второй категории

In [ ]:
train_table = pd.DataFrame({
    'category1': y1_train.values,
    'category2': y2_train.values,
})
train_table['embedding'] = list(X_train_emb)

cat2_models = {}       
cat2_single_answer = {} 

for category1 in train_table['category1'].unique():
    subset = train_table[train_table['category1'] == category1]
    unique_answers = subset['category2'].unique()

    if len(unique_answers) == 1:
        cat2_single_answer[category1] = unique_answers[0]
        continue

    smallest_group_size = subset['category2'].value_counts().min()
    n_neighbors = min(3, smallest_group_size)

    model = KNeighborsClassifier(weights='distance', n_neighbors=n_neighbors, metric='euclidean')
    model.fit(list(subset['embedding']), subset['category2'])
    cat2_models[category1] = model

print(f'обучено под-классификаторов: {len(cat2_models)}')
print(f'категорий с единственным вариантом: {len(cat2_single_answer)}')


обучено под-классификаторов: 51
категорий с единственным вариантом: 7


In [ ]:
def predict_category2(embedding, predicted_category1):
    if predicted_category1 in cat2_single_answer:
        return cat2_single_answer[predicted_category1]

    if predicted_category1 in cat2_models:
        model = cat2_models[predicted_category1]
        return model.predict([embedding])[0]

    return 'нет модели'


y2_pred = [predict_category2(emb, cat1) for emb, cat1 in zip(X_test_emb, y1_pred)]


оценка

In [ ]:
y1_test_list = list(y1_test)
y2_test_list = list(y2_test)

category1_was_correct = []
both_were_correct = []

for i in range(len(y1_test_list)):
    cat1_ok = (y1_pred[i] == y1_test_list[i])
    cat2_ok = (y2_pred[i] == y2_test_list[i])
    category1_was_correct.append(cat1_ok)
    both_were_correct.append(cat1_ok and cat2_ok)

cat1_accuracy = sum(category1_was_correct) / len(category1_was_correct)
cascade_accuracy = sum(both_were_correct) / len(both_were_correct)

correct_idx = [i for i in range(len(y1_test_list)) if category1_was_correct[i]]
cat2_given_cat1_correct = sum(y2_pred[i] == y2_test_list[i] for i in correct_idx) / len(correct_idx)

print(f'Точность category1:                              {cat1_accuracy:.1%}')
print(f'Точность каскада (оба уровня верны):              {cascade_accuracy:.1%}')
print(f'Точность category2 | category1 предсказан верно:  {cat2_given_cat1_correct:.1%}')


Точность category1:                              90.4%
Точность каскада (оба уровня верны):              86.1%
Точность category2 | category1 предсказан верно:  95.2%
